# v07 - Per-slot JPEG training cache at 320 px (CPU)

Second step of the own-model leg (`docs/research/v06-own-model-design.md` revision 3.3,
decision D-006). Builds the training cache from the v06 series choice. Runs on CPU.

Per study and slot (5 slots):

1. Re-read the chosen series' headers and order slices by `ImagePositionPatient`
   projected on the slice normal (same function as v06; fallback `InstanceNumber`).
2. Keep up to 32 slices: all slices when the series has at most 32, otherwise
   `linspace(0, N-1, 32)` rounded.
3. Decode, apply rescale, take one 0.5-99.5 percentile window per series over the cropped
   field, crop 140 mm around the image centre (zero padding if the field is smaller),
   resize to 320 x 320.
4. **Canonical orientation from geometry:** columns point medial (coronal, axial) or
   posterior (sagittal); rows point inferior (sagittal, coronal) or posterior (axial);
   slice order runs lateral to medial (sagittal), anterior to posterior (coronal) and
   superior to inferior (axial). The knee side comes from the `Laterality` tag, else from
   the sign of the image-centre x coordinate (a study within 20 mm of the midline is
   unresolved and treated as a left knee, flagged).
5. Encode each slice as JPEG quality 92. Test-time inference must apply the same function
   and the same encode/decode round trip.

`DRY_RUN = True` processes the first 50 studies and projects the full size and time;
set it to `False` for the full run.

Inputs: the competition data and the output of `lingxd/v06-dicom-audit`.
Outputs in `/kaggle/working/v07/`: `cache/<StudyInstanceUID>.npz` (concatenated JPEG
bytes with offsets, slot and raw-slice index per stored slice), `v07_cache_index.csv`
(per study and slot), `v07_cache_summary.json`, and preview montages.

In [ ]:
# v07 configuration
import json
import os
import time
from pathlib import Path

VERSION = 'v07'
WORK = Path('/kaggle/working') / VERSION
CACHE_DIR = WORK / 'cache'
PREVIEW_DIR = WORK / 'preview'
for d in (WORK, CACHE_DIR, PREVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)
T0 = time.time()

DRY_RUN = True
DRY_RUN_STUDIES = 50
N_WORKERS = max(1, os.cpu_count() or 1)

SLOTS = ['SAG_FS', 'SAG_NFS', 'COR_FS', 'COR_OTHER', 'AX_FS']
SLOT_PLANE = {'SAG_FS': 'Sagittal', 'SAG_NFS': 'Sagittal', 'COR_FS': 'Coronal',
              'COR_OTHER': 'Coronal', 'AX_FS': 'Axial'}
MAX_SLICES = 32
CROP_MM = 140.0
OUT_PX = 320
WINDOW_PCT = (0.5, 99.5)
JPEG_QUALITY = 92
SIDE_DEADZONE_MM = 20.0
OUTPUT_LIMIT_GB = 19.0   # projected full-run cache must stay below this
N_PREVIEW = 8
print(f'{VERSION}: workers={N_WORKERS}, dry_run={DRY_RUN}', flush=True)

In [ ]:
# Locate inputs without walking the competition image tree.
import pandas as pd

def find_competition_root():
    for p in (Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection')):
        if (p / 'train_series.csv').is_file():
            return p
    raise FileNotFoundError('competition data not attached')

def find_unique_file(name, root=Path('/kaggle/input')):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1:
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0]

COMP = find_competition_root()
IMAGE_ROOT = COMP / 'train_series'
series_audit = pd.read_csv(find_unique_file('v06_series_audit.csv'),
                           dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
v06_selection = pd.read_csv(find_unique_file('v06_slot_selection.csv'),
                            dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
studies = sorted(v06_selection.StudyInstanceUID.unique())
if DRY_RUN:
    studies = studies[:DRY_RUN_STUDIES]
print(f'competition root: {COMP}; studies to cache: {len(studies)}')

## 1. Shared functions

`order_slices` and `select_slots` are copied verbatim from v06; the next cell checks that
`select_slots` reproduces the v06 choice exactly. `prepare_study` is the function test-time
inference must reuse unchanged.

In [ ]:
import cv2
import numpy as np
import pydicom

HEADER_TAGS = ['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
               'PixelSpacing', 'Rows', 'Columns', 'Laterality']

def _floats(value, n):
    try:
        out = [float(v) for v in value]
    except Exception:
        return None
    return out if len(out) == n and all(np.isfinite(out)) else None

def order_slices(records):
    # records: list of dicts with name, ipp, iop, inst. Returns (ordered names, method, info).
    info = {}
    with_geom = [r for r in records if r['ipp'] is not None and r['iop'] is not None]
    if records and len(with_geom) == len(records):
        iop = np.asarray(with_geom[0]['iop'], dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])
        norm = np.linalg.norm(normal)
        if norm > 1e-6:
            normal = normal / norm
            pos = np.asarray([np.dot(r['ipp'], normal) for r in with_geom])
            info['n_unique_positions'] = int(len(np.unique(np.round(pos, 3))))
            info['obliquity_deg'] = float(np.degrees(np.arccos(min(1.0, np.abs(normal).max()))))
            iop_all = np.asarray([r['iop'] for r in with_geom], dtype=np.float64)
            info['iop_consistent'] = bool(np.allclose(iop_all, iop_all[0], atol=1e-3))
            if info['n_unique_positions'] >= 0.9 * len(records):
                order = np.argsort(pos, kind='stable')
                gaps = np.diff(pos[order])
                info['slice_gap_mm'] = float(np.median(np.abs(gaps))) if len(gaps) else float('nan')
                return [with_geom[i]['name'] for i in order], 'geometry', info
    if records and all(r['inst'] is not None for r in records):
        ordered = sorted(records, key=lambda r: (r['inst'], r['name']))
        return [r['name'] for r in ordered], 'instance_number', info
    return sorted(r['name'] for r in records), 'filename', info

SLOT_SPEC = [
    ('SAG_FS', 'Sagittal', True, 8),
    ('SAG_NFS', 'Sagittal', False, 6),
    ('COR_FS', 'Coronal', True, 8),
    ('COR_OTHER', 'Coronal', None, 4),
    ('AX_FS', 'Axial', True, 6),
]

def _rank_key(c):
    return (0 if c['geometry_ok'] else 1, -int(c['n_readable'] or 0), c['SeriesInstanceUID'])

def select_slots(study_series):
    # study_series: list of dicts with SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive,
    # geometry_ok, n_readable. Returns {slot: (series uid or None, duplicate flag, fallback flag)}.
    usable = [s for s in study_series if (s.get('n_readable') or 0) > 0]
    chosen, used = {}, set()
    for slot, plane, fluid_pref, _k in SLOT_SPEC:
        same_plane = sorted((s for s in usable if s['Anatomical_Plane'] == plane), key=_rank_key)
        if not same_plane:
            chosen[slot] = (None, False, False)
            continue
        unused = [s for s in same_plane if s['SeriesInstanceUID'] not in used]
        if fluid_pref is None:
            preferred = sorted(unused, key=lambda s: (int(s['Fluid_Sensitive']),) + _rank_key(s))
        else:
            preferred = [s for s in unused if bool(s['Fluid_Sensitive']) == fluid_pref]
        if preferred:
            pick, duplicate, fallback = preferred[0], False, False
        elif unused:
            pick, duplicate, fallback = unused[0], False, True
        else:
            pick, duplicate, fallback = same_plane[0], True, True
        used.add(pick['SeriesInstanceUID'])
        chosen[slot] = (pick['SeriesInstanceUID'], duplicate, fallback)
    return chosen

# Canonical patient-space directions (DICOM LPS: +x left, +y posterior, +z superior).
# Per plane: (column direction, row direction, slice direction); 'medial' depends on side.
CANONICAL = {
    'Sagittal': ('posterior', 'inferior', 'medial'),
    'Coronal': ('medial', 'inferior', 'posterior'),
    'Axial': ('medial', 'posterior', 'inferior'),
}

def direction_vector(name, side):
    if name == 'medial':
        # The right knee's medial side faces the patient's left (+x), and vice versa.
        return np.array([1.0, 0.0, 0.0]) if side == 'R' else np.array([-1.0, 0.0, 0.0])
    return {'posterior': np.array([0.0, 1.0, 0.0]),
            'inferior': np.array([0.0, 0.0, -1.0])}[name]

def series_headers(series_dir):
    # Read headers of every file; return records plus first-file metadata.
    names = sorted(p.name for p in series_dir.iterdir() if p.suffix.lower() == '.dcm')
    records, meta = [], None
    for name in names:
        ds = pydicom.dcmread(str(series_dir / name), stop_before_pixels=True, force=True,
                             specific_tags=HEADER_TAGS)
        try:
            inst = int(float(str(getattr(ds, 'InstanceNumber', '')).strip()))
        except ValueError:
            inst = None
        rec = {'name': name,
               'ipp': _floats(getattr(ds, 'ImagePositionPatient', None) or [], 3),
               'iop': _floats(getattr(ds, 'ImageOrientationPatient', None) or [], 6),
               'inst': inst}
        records.append(rec)
        if meta is None:
            meta = {'spacing': _floats(getattr(ds, 'PixelSpacing', None) or [], 2),
                    'rows': getattr(ds, 'Rows', None), 'cols': getattr(ds, 'Columns', None),
                    'laterality': str(getattr(ds, 'Laterality', '') or '').strip().upper()}
    return records, meta

def image_centre_x(rec, meta):
    if rec['ipp'] is None or rec['iop'] is None or not meta['spacing'] or not meta['rows'] or not meta['cols']:
        return None
    ipp, iop = np.asarray(rec['ipp']), np.asarray(rec['iop'])
    centre = (ipp + iop[:3] * meta['spacing'][1] * float(meta['cols']) / 2
              + iop[3:] * meta['spacing'][0] * float(meta['rows']) / 2)
    return float(centre[0])

def decide_side(tags, centre_xs):
    # Returns (side, source). Tag first, then geometry with a dead zone.
    letters = [t[0] for t in tags if t and t[0] in ('L', 'R')]
    if letters:
        return max(set(letters), key=letters.count), 'tag'
    if centre_xs:
        m = float(np.median(centre_xs))
        if abs(m) >= SIDE_DEADZONE_MM:
            return ('R' if m < 0 else 'L'), 'geometry'
    return 'L', 'unresolved'

def geometry_side(centre_xs):
    if not centre_xs:
        return None
    m = float(np.median(centre_xs))
    return None if abs(m) < SIDE_DEADZONE_MM else ('R' if m < 0 else 'L')

def crop_resize(arr, spacing):
    # Centre crop of CROP_MM (zero padding if smaller), resize to OUT_PX.
    rows, cols = arr.shape
    want_r = int(round(CROP_MM / spacing[0])) if spacing else rows
    want_c = int(round(CROP_MM / spacing[1])) if spacing else cols
    out = np.zeros((want_r, want_c), dtype=np.float32)
    src_r0, src_c0 = max(0, (rows - want_r) // 2), max(0, (cols - want_c) // 2)
    dst_r0, dst_c0 = max(0, (want_r - rows) // 2), max(0, (want_c - cols) // 2)
    h, w = min(rows, want_r), min(cols, want_c)
    out[dst_r0:dst_r0 + h, dst_c0:dst_c0 + w] = arr[src_r0:src_r0 + h, src_c0:src_c0 + w]
    interp = cv2.INTER_AREA if min(want_r, want_c) >= OUT_PX else cv2.INTER_LINEAR
    return cv2.resize(out, (OUT_PX, OUT_PX), interpolation=interp), (h, w)

def encode_jpeg(img_u8):
    ok, buf = cv2.imencode('.jpg', img_u8, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
    if not ok:
        raise RuntimeError('JPEG encode failed')
    return buf.tobytes()

def decode_jpeg(data):
    return cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

def prepare_study(study, slot_series, image_root):
    # slot_series: {slot: SeriesInstanceUID or None}. Returns (per-slot dict, study info).
    # Per slot: {'jpeg': [bytes], 'raw_index': [int], 'n_series': int, ...} or None.
    uids = sorted({u for u in slot_series.values() if u})
    plane_of = {u: SLOT_PLANE[s] for s, u in slot_series.items() if u}
    headers, tags, centre_xs = {}, [], []
    for uid in uids:
        records, meta = series_headers(image_root / study / uid)
        ordered, method, _info = order_slices(records)
        by_name = {r['name']: r for r in records}
        headers[uid] = (ordered, method, meta, by_name)
        tags.append(meta['laterality'])
        mid = by_name[ordered[len(ordered) // 2]]
        cx = image_centre_x(mid, meta)
        if cx is not None:
            centre_xs.append(cx)
    side, side_source = decide_side(tags, centre_xs)
    info = {'side': side, 'side_source': side_source,
            'tag_side': next((t[0] for t in tags if t and t[0] in ('L', 'R')), ''),
            'geometry_side': geometry_side(centre_xs) or ''}

    stored = {}
    for uid in uids:
        ordered, method, meta, by_name = headers[uid]
        n = len(ordered)
        keep = list(range(n)) if n <= MAX_SLICES else \
            sorted(set(np.linspace(0, n - 1, MAX_SLICES).round().astype(int).tolist()))
        spacing = meta['spacing']
        crops, fields, kept, n_bad = [], [], [], 0
        for i in keep:
            try:
                ds = pydicom.dcmread(str(image_root / study / uid / ordered[i]), force=True)
                arr = ds.pixel_array.astype(np.float32)
                if arr.ndim != 2:
                    raise ValueError(f'unexpected shape {arr.shape}')
            except Exception:
                n_bad += 1   # an unreadable slice is skipped, not fatal for the study
                continue
            kept.append(i)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            arr = arr * slope + intercept
            img, (h, w) = crop_resize(arr, spacing)
            crops.append(img)
            rows, cols = arr.shape
            r0, c0 = max(0, (rows - h) // 2), max(0, (cols - w) // 2)
            fields.append(arr[r0:r0 + h, c0:c0 + w].ravel())
        if not kept:
            stored[uid] = None
            continue
        lo, hi = np.percentile(np.concatenate(fields), WINDOW_PCT)
        # Orientation from the first stored slice's geometry.
        rec = by_name[ordered[kept[0]]]
        flip_lr = flip_ud = reverse = False
        if method == 'geometry' and rec['iop'] is not None:
            iop = np.asarray(rec['iop'])
            row_dir, col_dir = iop[:3], iop[3:]   # along a row (columns increase), down a column (rows increase)
            normal = np.cross(row_dir, col_dir)
            plane = plane_of[uid]
            want_cols, want_rows, want_slices = (direction_vector(n_, side) for n_ in CANONICAL[plane])
            flip_lr = float(np.dot(row_dir, want_cols)) < 0
            flip_ud = float(np.dot(col_dir, want_rows)) < 0
            reverse = float(np.dot(normal, want_slices)) < 0
        imgs = []
        for img in crops:
            u8 = np.clip((img - lo) / max(hi - lo, 1e-6) * 255.0, 0, 255).astype(np.uint8)
            if flip_lr:
                u8 = u8[:, ::-1]
            if flip_ud:
                u8 = u8[::-1, :]
            imgs.append(np.ascontiguousarray(u8))
        raw_index = list(kept)
        if reverse:
            imgs, raw_index = imgs[::-1], raw_index[::-1]
        stored[uid] = {'jpeg': [encode_jpeg(x) for x in imgs], 'raw_index': raw_index,
                       'n_series': n, 'sort_method': method, 'flip_lr': flip_lr,
                       'flip_ud': flip_ud, 'reverse': reverse, 'fov_ok': bool(spacing),
                       'n_decode_errors': n_bad,
                       'window': (float(lo), float(hi))}
    per_slot = {slot: (stored.get(uid) if uid else None) for slot, uid in slot_series.items()}
    return per_slot, info

## 2. Check the copied selection rule against v06

In [ ]:
by_study = {k: g.to_dict('records') for k, g in series_audit.groupby('StudyInstanceUID')}
v06_choice = {(r.StudyInstanceUID, r.slot): (r.SeriesInstanceUID if isinstance(r.SeriesInstanceUID, str) else None)
              for r in v06_selection.itertuples()}
mismatch = 0
SLOT_SERIES = {}
for study in studies:
    chosen = select_slots(by_study.get(study, []))
    SLOT_SERIES[study] = {slot: chosen[slot][0] for slot in SLOTS}
    mismatch += sum(chosen[slot][0] != v06_choice.get((study, slot)) for slot in SLOTS)
assert mismatch == 0, f'select_slots differs from v06 on {mismatch} slots'
print(f'select_slots reproduces the v06 choice on {len(studies)} studies', flush=True)

## 3. Build the cache

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def cache_one(study):
    t = time.time()
    try:
        per_slot, info = prepare_study(study, SLOT_SERIES[study], IMAGE_ROOT)
    except Exception as exc:
        return {'StudyInstanceUID': study, 'error': f'{type(exc).__name__}: {str(exc)[:300]}'}, []
    blobs, offsets, slot_idx, raw_idx = [], [0], [], []
    rows = []
    for s, slot in enumerate(SLOTS):
        entry = per_slot[slot]
        if entry is None:
            rows.append({'StudyInstanceUID': study, 'slot': slot, 'present': False})
            continue
        for b, r in zip(entry['jpeg'], entry['raw_index']):
            blobs.append(b)
            offsets.append(offsets[-1] + len(b))
            slot_idx.append(s)
            raw_idx.append(r)
        rows.append({'StudyInstanceUID': study, 'slot': slot, 'present': True,
                     'SeriesInstanceUID': SLOT_SERIES[study][slot],
                     'n_series_slices': entry['n_series'], 'n_stored': len(entry['jpeg']),
                     'bytes': sum(len(b) for b in entry['jpeg']),
                     'sort_method': entry['sort_method'], 'flip_lr': entry['flip_lr'],
                     'flip_ud': entry['flip_ud'], 'reverse': entry['reverse'],
                     'n_decode_errors': entry['n_decode_errors'],
                     'window_lo': entry['window'][0], 'window_hi': entry['window'][1]})
    np.savez(CACHE_DIR / f'{study}.npz',
             jpeg=np.frombuffer(b''.join(blobs), dtype=np.uint8),
             offsets=np.asarray(offsets, dtype=np.int64),
             slot=np.asarray(slot_idx, dtype=np.int8),
             raw_index=np.asarray(raw_idx, dtype=np.int16))
    summary = {'StudyInstanceUID': study, 'error': '', 'seconds': time.time() - t,
               'file_bytes': (CACHE_DIR / f'{study}.npz').stat().st_size, **info}
    return summary, rows

t = time.time()
study_rows, slot_rows = [], []
with ProcessPoolExecutor(N_WORKERS) as pool:
    for i, (summary, rows) in enumerate(pool.map(cache_one, studies, chunksize=2)):
        study_rows.append(summary)
        slot_rows.extend(rows)
        if (i + 1) % 200 == 0:
            print(f'  cached {i + 1}/{len(studies)} studies, {time.time() - t:.0f}s', flush=True)
wall = time.time() - t
study_index = pd.DataFrame(study_rows)
slot_index = pd.DataFrame(slot_rows)
errors = study_index[study_index.error.fillna('') != '']
print(f'cached {len(study_index)} studies in {wall:.0f}s; errors: {len(errors)}', flush=True)
print(errors.head(10).to_string())

## 4. Checks, projection and previews

In [ ]:
ok = study_index[study_index.error.fillna('') == '']
total_bytes = float(ok.file_bytes.sum())
n_full = int(v06_selection.StudyInstanceUID.nunique())
projected_gb = total_bytes / max(len(ok), 1) * n_full / 1e9
projected_hours = wall / max(len(ok), 1) * n_full / 3600
tagged = ok[(ok.tag_side != '') & (ok.geometry_side != '')]
side_agreement = float((tagged.tag_side == tagged.geometry_side).mean()) if len(tagged) else float('nan')

# Round-trip determinism: re-encoding a decoded slice must not be required for equality;
# check that decoding the stored bytes twice gives identical arrays.
probe = np.load(CACHE_DIR / f'{ok.StudyInstanceUID.iloc[0]}.npz')
b0 = probe['jpeg'][probe['offsets'][0]:probe['offsets'][1]].tobytes()
assert np.array_equal(decode_jpeg(b0), decode_jpeg(b0))

present = slot_index[slot_index.present]
summary = {
    'version': VERSION,
    'dry_run': DRY_RUN,
    'studies_cached': int(len(ok)),
    'errors': int(len(errors)),
    'cache_gb': round(total_bytes / 1e9, 3),
    'projected_full_gb': round(projected_gb, 2),
    'projected_full_hours': round(projected_hours, 2),
    'within_output_limit': bool(projected_gb < OUTPUT_LIMIT_GB),
    'stored_slices_per_slot': present.groupby('slot').n_stored.median().to_dict(),
    'mean_kb_per_slice': round(total_bytes / max(int(present.n_stored.sum()), 1) / 1024, 1),
    'side_source': ok.side_source.value_counts().to_dict(),
    'tag_vs_geometry_agreement': side_agreement,
    'tagged_with_geometry': int(len(tagged)),
    'flip_rates': present[['flip_lr', 'flip_ud', 'reverse']].mean().round(3).to_dict(),
    'sort_methods': present.sort_method.value_counts().to_dict(),
    'slice_decode_errors': int(present.n_decode_errors.sum()),
    'slots_lost_to_decode_errors': int((~slot_index.present).sum()),
    'opencv': cv2.__version__,
    'pydicom': pydicom.__version__,
    'elapsed_seconds': round(time.time() - T0, 1),
}
print(json.dumps(summary, indent=2))

# Preview montages: for a few studies, the middle stored slice of every slot.
for study in ok.StudyInstanceUID.head(N_PREVIEW):
    data = np.load(CACHE_DIR / f'{study}.npz')
    tiles = []
    for s, slot in enumerate(SLOTS):
        idx = np.flatnonzero(data['slot'] == s)
        if len(idx) == 0:
            tiles.append(np.zeros((OUT_PX, OUT_PX), np.uint8))
            continue
        k = idx[len(idx) // 2]
        tiles.append(decode_jpeg(data['jpeg'][data['offsets'][k]:data['offsets'][k + 1]].tobytes()))
    side = ok.set_index('StudyInstanceUID').side[study]
    montage = np.concatenate(tiles, axis=1)
    cv2.putText(montage, f'{study[-8:]} side={side}', (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 255, 1)
    cv2.imwrite(str(PREVIEW_DIR / f'v07_preview_{study[-12:]}.png'), montage)

In [ ]:
study_index.to_csv(WORK / 'v07_cache_studies.csv', index=False)
slot_index.to_csv(WORK / 'v07_cache_index.csv', index=False)
(WORK / 'v07_cache_summary.json').write_text(json.dumps(summary, indent=2))
print('written:', sorted(p.name for p in WORK.iterdir()))